# Two-year route forecast

One forecast of every route-month from an origin T through T+23, made once at T and never updated with actuals from inside that window. The unit is `OD_PAX` on one origin–destination route in one departure month. The network total is the sum of the route forecasts.

Primary origin: T = 2024-08, scored on 2024-08 … 2026-07. Stability origin: T = 2023-08. The unscored forward origin is T = 2026-08.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/aa-mplconfig")

import sys
import json
import subprocess

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import statsmodels
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

print("executable:", sys.executable)
print("python", sys.version.split()[0])
print("pandas", pd.__version__)
print("numpy", np.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("statsmodels", statsmodels.__version__)

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")
airports = pd.read_csv(root / "data" / "raw" / "AA_AIRPORT_INFO.csv")
pax["DATE"] = pd.to_datetime({"year": pax["DEP_YEAR"], "month": pax["DEP_MONTH"], "day": 1})

expected_months = pd.date_range("2019-01-01", "2026-07-01", freq="MS")
keys = ["ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH"]
if pax.duplicated(keys).any():
    raise SystemExit("The route-month key is not unique. Stop and ask before summing duplicates.")
assert len(pax) == 1_533_522
assert set(pax["DATE"].unique()) == set(expected_months)
assert pax["OD_PAX"].isna().sum() == 0
assert pax["OD_PAX"].min() >= 0
print(f"rows: {len(pax):,}")
print(f"months: {pax['DATE'].nunique()}")
print(f"duplicate keys: {int(pax.duplicated(keys).sum())}")


## 2. Features

`decimal_degrees`, `airport_lookup`, and `build_features` are copied from `pri-biswal.ipynb`. Only the original passenger columns go into `build_features`. Training rows and forecast-set rows are concatenated first so the categories match.


In [ ]:
# Copied verbatim from notebooks/pri-biswal.ipynb.
def decimal_degrees(degrees, minutes, seconds, hemisphere):
    magnitude = degrees.astype(float) + minutes.astype(float) / 60 + seconds.astype(float) / 3600
    sign = hemisphere.astype(str).str.upper().map({"N": 1, "S": -1, "E": 1, "W": -1})
    return magnitude * sign


def airport_lookup(airports):
    """One row per airport. Names and raw clock-position fields are not features."""
    # .to_numpy() keeps each column in row order. Passing the airport code as
    # the DataFrame index would realign those columns onto the codes and blank them.
    lookup = pd.DataFrame(
        {
            "CITY_METRO_IATA_CD": airports["CITY_METRO_IATA_CD"].fillna("MISSING").to_numpy(),
            "STATE_PROVNC_CD": airports["STATE_PROVNC_CD"].fillna("MISSING").to_numpy(),
            "CNTRY_CD": airports["CNTRY_CD"].fillna("MISSING").to_numpy(),
            "WRLD_AREA_DOT_CD": airports["WRLD_AREA_DOT_CD"].astype("int64").astype(str).to_numpy(),
            "LAT": decimal_degrees(
                airports["LAT_DEGREE_QTY"],
                airports["LAT_MINUTE_QTY"],
                airports["LAT_SECOND_QTY"],
                airports["LAT_HEMSPHR_CD"],
            ).to_numpy(),
            "LNG": decimal_degrees(
                airports["LNGTD_DEGREE_QTY"],
                airports["LNGTD_MINUTE_QTY"],
                airports["LNGTD_SECOND_QTY"],
                airports["LNGTD_HEMSPHR_CD"],
            ).to_numpy(),
            # A runway length of 0 is not a real runway. None of those airports
            # appear on the forecasting routes; this keeps the lookup honest.
            "RUNWAY_FT": airports["LNGST_RUNWAY_FT_QTY"].replace(0, np.nan).to_numpy(),
            "ELEVATION_FT": airports["ELEVATN_FT_QTY"].to_numpy(),
        }
    )
    lookup.index = airports["AIRPRT_CD"].to_numpy()
    return lookup[~lookup.index.duplicated()]


def build_features(pax, airports):
    lookup = airport_lookup(airports)
    frame = pax.drop(columns=["Unnamed: 0", "SF_LOAD_TMS", "OD_REV_USD"]).copy()
    y = frame.pop("OD_PAX")
    month = frame["DEP_MONTH"].astype(int)
    features = frame.join(lookup.add_prefix("ORG_"), on="ORG_AIRPRT")
    features = features.join(lookup.add_prefix("DES_"), on="DES_AIRPRT")
    features["DEP_MONTH"] = month.astype(str)
    features["MONTH_SIN"] = np.sin(2 * np.pi * month / 12)
    features["MONTH_COS"] = np.cos(2 * np.pi * month / 12)
    categorical = [
        "DEP_MONTH",
        "ORG_AIRPRT",
        "DES_AIRPRT",
        "ORG_CITY_METRO_IATA_CD",
        "DES_CITY_METRO_IATA_CD",
        "ORG_STATE_PROVNC_CD",
        "DES_STATE_PROVNC_CD",
        "ORG_CNTRY_CD",
        "DES_CNTRY_CD",
        "ORG_WRLD_AREA_DOT_CD",
        "DES_WRLD_AREA_DOT_CD",
    ]
    for column in categorical:
        features[column] = features[column].astype("category")
    return features, y


ORIGINAL = [c for c in pax.columns if c != "DATE"]
BANNED = {"OD_PAX", "OD_REV_USD", "Unnamed: 0", "SF_LOAD_TMS", "DATE", "NAIVE_PAX", "HORIZON_YEAR"}


def forecast_set(pax, T):
    """Route-months in the 12 months before T, moved forward 12 and 24 months."""
    T = pd.Timestamp(T)
    start = T - pd.DateOffset(months=12)
    history = pax.loc[(pax["DATE"] >= start) & (pax["DATE"] < T)].copy()
    pieces = []
    for shift, horizon_year in ((12, 1), (24, 2)):
        part = history.copy()
        shifted = part["DATE"] + pd.DateOffset(months=shift)
        part["DEP_YEAR"] = shifted.dt.year.astype(int)
        part["DEP_MONTH"] = shifted.dt.month.astype(int)
        part["DATE"] = shifted
        part["NAIVE_PAX"] = history["OD_PAX"].to_numpy()
        part["HORIZON_YEAR"] = horizon_year
        pieces.append(part)
    frame = pd.concat(pieces, ignore_index=True)
    assert len(frame) == 2 * len(history)
    assert not frame.duplicated(keys).any()
    return frame


def feature_matrices(pax, airports, T):
    """Training rows before T, then the forecast set, in one concat."""
    T = pd.Timestamp(T)
    history = pax.loc[pax["DATE"] < T, ORIGINAL]
    frame = forecast_set(pax, T)
    future = frame.loc[:, ORIGINAL].copy()
    future["OD_PAX"] = np.nan
    combined = pd.concat([history, future], ignore_index=True)
    features, target = build_features(combined, airports)
    n_train = len(history)
    X_train = features.iloc[:n_train].copy()
    y_train = target.iloc[:n_train].copy()
    X_future = features.iloc[n_train:].copy()
    train_dates = pd.to_datetime(
        {"year": X_train["DEP_YEAR"].astype(int), "month": X_train["DEP_MONTH"].astype(int), "day": 1}
    )
    assert train_dates.max() < T
    assert y_train.notna().all()
    assert len(X_future) == len(frame)
    return X_train, y_train, X_future, frame


X_train, y_train, X_future, future_frame = feature_matrices(pax, airports, "2024-08-01")
assert X_train.shape[1] == 22
assert X_train.select_dtypes("category").shape[1] == 11
assert BANNED.isdisjoint(X_train.columns)
print(f"training rows before 2024-08: {len(X_train):,}")
print(f"forecast-set rows: {len(X_future):,}")
print(f"columns: {X_train.shape[1]}")
print(f"categorical columns: {X_train.select_dtypes('category').shape[1]}")


## 3. Evaluation

An absent actual counts as 0 passengers. A route-month outside the forecast set counts as forecast 0. Metrics are compared within one grain.


In [ ]:
def score(frame, pax, T):
    """Outer-join a forecast frame to the actual rows in [T, T+24)."""
    T = pd.Timestamp(T)
    end = T + pd.DateOffset(months=24)
    actual = pax.loc[(pax["DATE"] >= T) & (pax["DATE"] < end), keys + ["OD_PAX"]]
    actual_total = float(actual["OD_PAX"].sum())
    merged = frame.loc[:, keys + ["FORECAST"]].merge(actual, on=keys, how="outer")
    merged["FORECAST"] = merged["FORECAST"].fillna(0.0)
    merged["OD_PAX"] = merged["OD_PAX"].fillna(0.0)
    assert abs(float(merged["OD_PAX"].sum()) - actual_total) < 1e-4
    merged["DATE"] = pd.to_datetime(
        {"year": merged["DEP_YEAR"].astype(int), "month": merged["DEP_MONTH"].astype(int), "day": 1}
    )
    merged["h"] = (merged["DATE"].dt.year - T.year) * 12 + (merged["DATE"].dt.month - T.month) + 1
    assert merged["h"].between(1, 24).all()
    return merged


def _wape(actual, forecast):
    return float(np.abs(forecast - actual).sum() / actual.sum())


def _bias(actual, forecast):
    return float((forecast.sum() - actual.sum()) / actual.sum())


def metric_block(scored):
    actual = scored["OD_PAX"].to_numpy()
    forecast = scored["FORECAST"].to_numpy()
    out = {}
    masks = {
        "all": np.ones(len(scored), dtype=bool),
        "y1": scored["h"].to_numpy() <= 12,
        "y2": scored["h"].to_numpy() >= 13,
    }
    for name, mask in masks.items():
        out[f"route_wape_{name}"] = _wape(actual[mask], forecast[mask])
        out[f"bias_{name}"] = _bias(actual[mask], forecast[mask])
        out[f"route_mae_{name}"] = float(np.mean(np.abs(forecast[mask] - actual[mask])))
        out[f"route_rmse_{name}"] = float(np.sqrt(np.mean((forecast[mask] - actual[mask]) ** 2)))
    monthly = scored.groupby("DATE", as_index=False)[["OD_PAX", "FORECAST"]].sum().sort_values("DATE")
    monthly["h"] = scored.groupby("DATE")["h"].min().reindex(monthly["DATE"]).to_numpy()
    error = monthly["FORECAST"].to_numpy() - monthly["OD_PAX"].to_numpy()
    out["network_mae"] = float(np.mean(np.abs(error)))
    out["network_rmse"] = float(np.sqrt(np.mean(error ** 2)))
    out["network_wape"] = _wape(monthly["OD_PAX"].to_numpy(), monthly["FORECAST"].to_numpy())
    by_route = scored.groupby(["ORG_AIRPRT", "DES_AIRPRT"], as_index=False)[["OD_PAX", "FORECAST"]].sum()
    out["route_total_wape"] = _wape(by_route["OD_PAX"].to_numpy(), by_route["FORECAST"].to_numpy())
    return out, monthly


def segment_labels(pax, airports, T):
    T = pd.Timestamp(T)
    start = T - pd.DateOffset(months=12)
    history = pax.loc[(pax["DATE"] >= start) & (pax["DATE"] < T)]
    volume = history.groupby(["ORG_AIRPRT", "DES_AIRPRT"])["OD_PAX"].sum()
    quartile = pd.qcut(volume, 4, labels=["Q1", "Q2", "Q3", "Q4"])
    country = airport_lookup(airports)["CNTRY_CD"]
    return quartile, country


def attach_segments(scored, quartile, country):
    labeled = scored.copy()
    index = pd.MultiIndex.from_frame(labeled[keys[:2]])
    looked = quartile.reindex(index)
    labeled["quartile"] = np.where(looked.isna(), "new", looked.astype(str))
    origin_country = labeled["ORG_AIRPRT"].map(country)
    destination_country = labeled["DES_AIRPRT"].map(country)
    labeled["market"] = np.where(
        (origin_country == "US") & (destination_country == "US"),
        "domestic",
        "international",
    )
    labeled["year"] = np.where(labeled["h"] <= 12, 1, 2)
    return labeled


def grouped_wape(labeled, column):
    rows = []
    for (level, year), part in labeled.groupby([column, "year"], observed=True):
        rows.append(
            {
                column: str(level),
                "year": int(year),
                "wape": _wape(part["OD_PAX"].to_numpy(), part["FORECAST"].to_numpy()),
            }
        )
    return rows


def wape_by_horizon(scored):
    rows = []
    for horizon, part in scored.groupby("h"):
        rows.append(
            {
                "h": int(horizon),
                "wape": _wape(part["OD_PAX"].to_numpy(), part["FORECAST"].to_numpy()),
            }
        )
    return rows


## 4. Seasonal naive

Each forecast route-month receives that route's passengers in the same calendar month of the year before T, for both forecast years. Summed over routes, the T = 2024-08 network series is the PR #77 seasonal naive.


In [ ]:
ORIGINS = (pd.Timestamp("2024-08-01"), pd.Timestamp("2023-08-01"))
matrices = {}
naive_scored = {}
naive_metrics = {}
naive_monthly = {}

for origin in ORIGINS:
    label = origin.strftime("%Y-%m")
    if origin == pd.Timestamp("2024-08-01"):
        matrices[label] = (X_train, y_train, X_future, future_frame)
    else:
        matrices[label] = feature_matrices(pax, airports, origin)
    _, _, _, frame = matrices[label]
    predicted = frame[keys].copy()
    predicted["FORECAST"] = frame["NAIVE_PAX"].to_numpy()
    scored = score(predicted, pax, origin)
    metrics, monthly = metric_block(scored)
    naive_scored[label] = scored
    naive_metrics[label] = metrics
    naive_monthly[label] = monthly
    print(label, "naive network MAE", round(metrics["network_mae"], 2), "RMSE", round(metrics["network_rmse"], 2))
    print(label, "naive route WAPE", round(metrics["route_wape_all"], 4), "y1", round(metrics["route_wape_y1"], 4), "y2", round(metrics["route_wape_y2"], 4))

assert abs(naive_metrics["2024-08"]["network_mae"] - 3_113_174.10) < 1
assert abs(naive_metrics["2024-08"]["network_rmse"] - 4_319_584.87) < 1
print("PR #77 network naive reproduced within 1 passenger")


## 5. Gradient-boosted tree

Settings were fixed before this run: `HistGradientBoostingRegressor(loss="poisson", max_iter=1000, early_stopping=True, random_state=9)`. Every other argument stays at the scikit-learn default. The tree is carried forward only if its WAPE is lower than the naive on all six checks: route-month year 1, route-month year 2, and network monthly, at both origins.


In [ ]:
def fit_gbt(features, target, loss="poisson", quantile=None):
    arguments = dict(loss=loss, max_iter=1000, early_stopping=True, random_state=9)
    if quantile is not None:
        arguments["quantile"] = quantile
    model = HistGradientBoostingRegressor(**arguments)
    model.fit(features, target)
    return model


settings_changes = []
gbt_scored = {}
gbt_metrics = {}
gbt_monthly = {}
gbt_models = {}
gbt_iterations = {}

for origin in ORIGINS:
    label = origin.strftime("%Y-%m")
    X_origin, y_origin, X_cast, frame = matrices[label]
    train_dates = pd.to_datetime(
        {"year": X_origin["DEP_YEAR"].astype(int), "month": X_origin["DEP_MONTH"].astype(int), "day": 1}
    )
    assert train_dates.max() < origin
    model = fit_gbt(X_origin, y_origin)
    gbt_iterations[label] = int(model.n_iter_)
    print(label, "iterations", model.n_iter_)
    predicted = frame[keys].copy()
    predicted["FORECAST"] = model.predict(X_cast)
    scored = score(predicted, pax, origin)
    metrics, monthly = metric_block(scored)
    gbt_scored[label] = scored
    gbt_metrics[label] = metrics
    gbt_monthly[label] = monthly
    if label == "2024-08":
        gbt_models[label] = model
    print(label, "gbt route WAPE", round(metrics["route_wape_all"], 4), "network WAPE", round(metrics["network_wape"], 4))

CHECKS = (
    ("route_wape_y1", "route-month WAPE, year 1"),
    ("route_wape_y2", "route-month WAPE, year 2"),
    ("network_wape", "network monthly WAPE"),
)
bar_rows = []
for label in ("2023-08", "2024-08"):
    for key, title in CHECKS:
        naive_value = naive_metrics[label][key]
        gbt_value = gbt_metrics[label][key]
        bar_rows.append(
            {
                "origin": label,
                "check": title,
                "naive": naive_value,
                "gbt": gbt_value,
                "gbt_lower": bool(gbt_value < naive_value),
            }
        )
bar = pd.DataFrame(bar_rows)
carried = "gbt" if bar["gbt_lower"].all() else "naive"
print(bar.assign(naive=lambda d: (d["naive"] * 100).round(2), gbt=lambda d: (d["gbt"] * 100).round(2)).to_string(index=False))
print("carried model:", carried)
print("settings changes:", settings_changes)


## Decision

The seasonal naive is carried forward. The gradient-boosted tree was lower on two of the six checks, both at the August 2023 origin, and higher on the other four, including every check at August 2024. The rule fixed before scoring carries the tree only when it is lower on all six. It is scored and rejected.

Both fits ran to the 1,000-iteration cap. Early stopping did not end them. No setting was changed after the scores were seen.


## 6. Diagnostics

Error by horizon, segment, and month is computed for both models. The network band uses the carried model's monthly percent errors, pooled across the two origins within each horizon year. Percent error is (forecast − actual) / actual. Quantile fits and permutation importance run only if the gradient-boosted tree is carried.


In [ ]:
segment_cache = {}
horizon_rows = []
segment_rows = {"quartile": [], "market": []}
new_share = {}

for label in ("2024-08", "2023-08"):
    quartile, country = segment_labels(pax, airports, label + "-01")
    for model_name, store in (("naive", naive_scored), ("gbt", gbt_scored)):
        labeled = attach_segments(store[label], quartile, country)
        for row in wape_by_horizon(store[label]):
            horizon_rows.append({"origin": label, "model": model_name, **row})
        for kind in ("quartile", "market"):
            for row in grouped_wape(labeled, kind):
                segment_rows[kind].append({"origin": label, "model": model_name, **row})
        if model_name == "naive":
            new_share[label] = float(
                labeled.loc[labeled["quartile"] == "new", "OD_PAX"].sum() / labeled["OD_PAX"].sum()
            )
    segment_cache[label] = (quartile, country)

band_frames = []
for label in ("2023-08", "2024-08"):
    monthly = (gbt_monthly if carried == "gbt" else naive_monthly)[label].copy()
    monthly["pct"] = (monthly["FORECAST"] - monthly["OD_PAX"]) / monthly["OD_PAX"]
    monthly["year"] = np.where(monthly["h"] <= 12, 1, 2)
    monthly["origin"] = label
    band_frames.append(monthly)
band_pool = pd.concat(band_frames, ignore_index=True)
band = (
    band_pool.groupby("year")["pct"]
    .quantile([0.1, 0.9])
    .unstack()
    .rename(columns={0.1: "p10", 0.9: "p90"})
)
print("network band percent errors")
print((band * 100).round(2).to_string())
print("share of actual passengers on routes outside the forecast set")
for label, share in new_share.items():
    print(label, round(share * 100, 2))

coverage = None
importance_rows = None
if carried == "gbt":
    X_origin, y_origin, X_cast, frame = matrices["2024-08"]
    quantile_models = {}
    quantile_prediction = {}
    for level, rate in (("p10", 0.1), ("p90", 0.9)):
        quantile_model = fit_gbt(X_origin, y_origin, loss="quantile", quantile=rate)
        quantile_models[level] = int(quantile_model.n_iter_)
        quantile_prediction[level] = quantile_model.predict(X_cast)
        print("quantile", level, "iterations", quantile_model.n_iter_)
    interval = frame[keys].copy()
    interval["p10"] = quantile_prediction["p10"]
    interval["p90"] = quantile_prediction["p90"]
    crossed = interval["p10"] > interval["p90"]
    crossing_rate = float(crossed.mean())
    interval["lo"] = np.minimum(interval["p10"], interval["p90"])
    interval["hi"] = np.maximum(interval["p10"], interval["p90"])
    actual = pax.loc[
        (pax["DATE"] >= "2024-08-01") & (pax["DATE"] < "2026-08-01"),
        keys + ["OD_PAX"],
    ]
    covered_rows = interval.merge(actual, on=keys, how="left")
    covered_rows["OD_PAX"] = covered_rows["OD_PAX"].fillna(0.0)
    covered_rows["hit"] = (covered_rows["OD_PAX"] >= covered_rows["lo"]) & (
        covered_rows["OD_PAX"] <= covered_rows["hi"]
    )
    covered_rows["DATE"] = pd.to_datetime(
        {
            "year": covered_rows["DEP_YEAR"].astype(int),
            "month": covered_rows["DEP_MONTH"].astype(int),
            "day": 1,
        }
    )
    covered_rows["h"] = (covered_rows["DATE"].dt.year - 2024) * 12 + (covered_rows["DATE"].dt.month - 8) + 1
    covered_rows["year"] = np.where(covered_rows["h"] <= 12, 1, 2)
    quartile, country = segment_cache["2024-08"]
    covered_rows = attach_segments(covered_rows, quartile, country)
    coverage = {
        "crossing_rate": crossing_rate,
        "overall": float(covered_rows["hit"].mean()),
        "by_year": {
            str(int(year)): float(part["hit"].mean())
            for year, part in covered_rows.groupby("year")
        },
        "by_quartile": {
            str(level): float(part["hit"].mean())
            for level, part in covered_rows.groupby("quartile")
        },
        "quantile_iterations": quantile_models,
        "negative_p10_share": float((interval["p10"] < 0).mean()),
    }
    print("P10/P90 crossing rate", round(crossing_rate, 4), "coverage", round(coverage["overall"], 4))

    sample = np.random.RandomState(9).choice(len(X_cast), size=100_000, replace=False)
    sample_actual = covered_rows["OD_PAX"].to_numpy()[sample]
    importance = permutation_importance(
        gbt_models["2024-08"],
        X_cast.iloc[sample],
        sample_actual,
        scoring="neg_mean_absolute_error",
        n_repeats=3,
        random_state=9,
    )
    importance_rows = (
        pd.DataFrame({"feature": X_cast.columns, "importance": importance.importances_mean})
        .sort_values("importance", ascending=False)
        .to_dict(orient="records")
    )
    print("top feature", importance_rows[0]["feature"], round(importance_rows[0]["importance"], 2))

visuals = root / "team-09-modeling" / "visuals"
visuals.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)
NAIVE_COLOR = "#b35c1e"
GBT_COLOR = "#1d4e89"


def save(fig, name):
    fig.tight_layout()
    fig.savefig(visuals / name, dpi=120)
    plt.close(fig)


history = (
    pax.loc[(pax["DATE"] >= "2022-08-01") & (pax["DATE"] <= "2026-07-01")]
    .groupby("DATE")["OD_PAX"]
    .sum()
    .sort_index()
)
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(history.index, history.to_numpy() / 1e6, color="black", label="Actual", linewidth=1.8)
axes[0].plot(
    naive_monthly["2024-08"]["DATE"],
    naive_monthly["2024-08"]["FORECAST"] / 1e6,
    color=NAIVE_COLOR,
    label="Seasonal naive",
    linewidth=1.6,
)
axes[0].plot(
    gbt_monthly["2024-08"]["DATE"],
    gbt_monthly["2024-08"]["FORECAST"] / 1e6,
    color=GBT_COLOR,
    label="Gradient-boosted tree",
    linewidth=1.6,
)
axes[0].axvline(pd.Timestamp("2024-08-01"), color="gray", linestyle="--", linewidth=1, label="Origin T")
axes[0].set_ylabel("Network passengers (millions)")
axes[0].set_title("Network passengers, August 2022 to July 2026")
axes[0].legend(frameon=False, ncol=2)
for model_name, store, color in (
    ("Seasonal naive", naive_monthly, NAIVE_COLOR),
    ("Gradient-boosted tree", gbt_monthly, GBT_COLOR),
):
    monthly = store["2024-08"]
    pct = (monthly["FORECAST"] - monthly["OD_PAX"]) / monthly["OD_PAX"] * 100
    axes[1].plot(monthly["DATE"], pct, color=color, label=model_name, linewidth=1.6)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Network error (%)")
axes[1].set_title("Monthly network error, August 2024 to July 2026")
axes[1].legend(frameon=False)
save(fig, "backtest-network.png")

horizon = pd.DataFrame(horizon_rows)
fig, ax = plt.subplots(figsize=(10, 5))
for model_name, color in (("naive", NAIVE_COLOR), ("gbt", GBT_COLOR)):
    for origin, style in (("2024-08", "-"), ("2023-08", "--")):
        part = horizon[(horizon["model"] == model_name) & (horizon["origin"] == origin)].sort_values("h")
        name = "Seasonal naive" if model_name == "naive" else "Gradient-boosted tree"
        ax.plot(
            part["h"],
            part["wape"] * 100,
            color=color,
            linestyle=style,
            label=f"{name}, {origin}",
            linewidth=1.6,
        )
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("Route-month WAPE (%)")
ax.set_title("Route-month error by horizon")
ax.set_xticks(range(1, 25, 2))
ax.legend(frameon=False, ncol=2)
save(fig, "error-by-horizon.png")

primary_segments = {
    kind: [row for row in rows if row["origin"] == "2024-08"] for kind, rows in segment_rows.items()
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, kind, title, order in (
    (axes[0], "quartile", "By route volume quartile", ["Q1", "Q2", "Q3", "Q4"]),
    (axes[1], "market", "By market", ["domestic", "international"]),
):
    table = pd.DataFrame(primary_segments[kind])
    x = np.arange(len(order))
    width = 0.18
    series = (
        ("naive", 1, NAIVE_COLOR, ""),
        ("gbt", 1, GBT_COLOR, ""),
        ("naive", 2, NAIVE_COLOR, "//"),
        ("gbt", 2, GBT_COLOR, "//"),
    )
    for offset, (model_name, year, color, hatch) in enumerate(series):
        values = []
        for level in order:
            match = table[
                (table["model"] == model_name) & (table["year"] == year) & (table[kind] == level)
            ]
            values.append(float(match["wape"].iloc[0]) * 100)
        label = ("Seasonal naive" if model_name == "naive" else "GBT") + f", year {year}"
        ax.bar(x + (offset - 1.5) * width, values, width=width, color=color, hatch=hatch, label=label)
    ax.set_xticks(x, order)
    ax.set_ylabel("Route-month WAPE (%)")
    ax.set_title(title)
axes[1].legend(frameon=False, fontsize=8)
fig.suptitle("Where the error sits, origin August 2024")
save(fig, "error-by-segment.png")

if importance_rows is not None:
    ordered = list(reversed(importance_rows))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([row["feature"] for row in ordered], [row["importance"] for row in ordered], color=GBT_COLOR)
    ax.set_xlabel("Increase in MAE when the column is shuffled")
    ax.set_title("What the tree uses on the August 2024 forecast set")
    save(fig, "feature-importance.png")
print("figures written")


## 7. Forward forecast

T = 2026-08 has no actuals. The carried model is refit on every realized row through July 2026. The route table stays in `data/processed/`. The chart is the network total.


In [ ]:
forward_origin = pd.Timestamp("2026-08-01")
X_forward, y_forward, X_cast, frame = feature_matrices(pax, airports, forward_origin)
train_dates = pd.to_datetime(
    {"year": X_forward["DEP_YEAR"].astype(int), "month": X_forward["DEP_MONTH"].astype(int), "day": 1}
)
assert train_dates.max() < forward_origin
assert len(frame) == 2 * int(((pax["DATE"] >= "2025-08-01") & (pax["DATE"] < forward_origin)).sum())

naive_forecast = frame["NAIVE_PAX"].to_numpy()
forward_iterations = None
quantile_iterations = None
if carried == "gbt":
    forward_model = fit_gbt(X_forward, y_forward)
    forward_iterations = int(forward_model.n_iter_)
    point = forward_model.predict(X_cast)
    print("forward iterations", forward_iterations)
    quantile_iterations = {}
    quantile_values = {}
    for level, rate in (("p10", 0.1), ("p90", 0.9)):
        quantile_model = fit_gbt(X_forward, y_forward, loss="quantile", quantile=rate)
        quantile_iterations[level] = int(quantile_model.n_iter_)
        quantile_values[level] = quantile_model.predict(X_cast)
        print("forward quantile", level, "iterations", quantile_model.n_iter_)
    low = np.minimum(quantile_values["p10"], quantile_values["p90"])
    high = np.maximum(quantile_values["p10"], quantile_values["p90"])
else:
    point = naive_forecast

table = frame[keys].copy()
table["FORECAST_PAX"] = point
if carried == "gbt":
    table["P10_PAX"] = low
    table["P90_PAX"] = high
processed = root / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
csv_path = processed / "forecast-2026-08-to-2028-07.csv"
table.to_csv(csv_path, index=False)
forecast_dates = pd.to_datetime(
    {"year": table["DEP_YEAR"].astype(int), "month": table["DEP_MONTH"].astype(int), "day": 1}
)
assert forecast_dates.min() == pd.Timestamp("2026-08-01")
assert forecast_dates.max() == pd.Timestamp("2028-07-01")
assert forecast_dates.nunique() == 24
assert int(table.isna().sum().sum()) == 0
ignored = subprocess.check_output(["git", "check-ignore", str(csv_path)], cwd=root, text=True).strip()
assert ignored.endswith("forecast-2026-08-to-2028-07.csv")

network = table.copy()
network["DATE"] = forecast_dates
network["h"] = (network["DATE"].dt.year - 2026) * 12 + (network["DATE"].dt.month - 8) + 1
monthly_forward = network.groupby(["DATE", "h"], as_index=False)["FORECAST_PAX"].sum().sort_values("DATE")
monthly_forward["year"] = np.where(monthly_forward["h"] <= 12, 1, 2)
monthly_forward["low"] = monthly_forward["FORECAST_PAX"] * (
    1 + monthly_forward["year"].map(band["p10"])
)
monthly_forward["high"] = monthly_forward["FORECAST_PAX"] * (
    1 + monthly_forward["year"].map(band["p90"])
)
assert len(monthly_forward) == 24
assert monthly_forward[["FORECAST_PAX", "low", "high"]].isna().sum().sum() == 0

full_history = pax.groupby("DATE")["OD_PAX"].sum().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(full_history.index, full_history.to_numpy() / 1e6, color="black", label="Actual", linewidth=1.6)
ax.plot(
    monthly_forward["DATE"],
    monthly_forward["FORECAST_PAX"] / 1e6,
    color=GBT_COLOR if carried == "gbt" else NAIVE_COLOR,
    label="Forecast",
    linewidth=1.6,
)
ax.fill_between(
    monthly_forward["DATE"],
    monthly_forward["low"] / 1e6,
    monthly_forward["high"] / 1e6,
    color="#9aa4b2",
    alpha=0.45,
    label="Approximate band",
    linewidth=0,
)
ax.axvline(pd.Timestamp("2026-08-01"), color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Network passengers (millions)")
ax.set_title("Network forecast, August 2026 to July 2028")
ax.legend(frameon=False)
save(fig, "forward-forecast.png")
print("forward year-1 total", round(float(monthly_forward.loc[monthly_forward["year"] == 1, "FORECAST_PAX"].sum()), 1))
print("forward year-2 total", round(float(monthly_forward.loc[monthly_forward["year"] == 2, "FORECAST_PAX"].sum()), 1))


In [ ]:
def pack_metrics(metrics):
    return {key: float(value) for key, value in metrics.items()}


def pack_monthly(naive, gbt):
    rows = []
    for naive_row, gbt_row in zip(naive.to_dict(orient="records"), gbt.to_dict(orient="records"), strict=True):
        rows.append(
            {
                "date": str(pd.Timestamp(naive_row["DATE"]).date()),
                "actual": float(naive_row["OD_PAX"]),
                "naive": float(naive_row["FORECAST"]),
                "gbt": float(gbt_row["FORECAST"]),
                "h": int(naive_row["h"]),
            }
        )
    return rows


summary = {
    "versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "matplotlib": matplotlib.__version__,
        "statsmodels": statsmodels.__version__,
    },
    "settings_changes": settings_changes,
    "gbt_params": gbt_models["2024-08"].get_params(),
    "gbt_iterations": gbt_iterations,
    "carried": carried,
    "bar": bar_rows,
    "naive": {label: pack_metrics(naive_metrics[label]) for label in naive_metrics},
    "gbt": {label: pack_metrics(gbt_metrics[label]) for label in gbt_metrics},
    "horizon": horizon_rows,
    "segments": segment_rows,
    "new_share": new_share,
    "monthly": {label: pack_monthly(naive_monthly[label], gbt_monthly[label]) for label in naive_monthly},
    "band": {str(int(year)): {"p10": float(row.p10), "p90": float(row.p90)} for year, row in band.iterrows()},
    "coverage": coverage,
    "importance": importance_rows,
    "forward": {
        "iterations": forward_iterations,
        "quantile_iterations": quantile_iterations,
        "year1": float(monthly_forward.loc[monthly_forward["year"] == 1, "FORECAST_PAX"].sum()),
        "year2": float(monthly_forward.loc[monthly_forward["year"] == 2, "FORECAST_PAX"].sum()),
        "monthly": [
            {
                "date": str(pd.Timestamp(row.DATE).date()),
                "forecast": float(row.FORECAST_PAX),
                "low": float(row.low),
                "high": float(row.high),
            }
            for row in monthly_forward.itertuples(index=False)
        ],
    },
}
summary_path = root / ".planning" / "run-summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("carried", carried)
print("summary", summary_path)
for label in ("2024-08", "2023-08"):
    print(label, "naive", {k: round(naive_metrics[label][k], 6) for k in ("route_wape_all", "route_wape_y1", "route_wape_y2", "network_wape", "bias_all", "network_mae", "network_rmse")})
    print(label, "gbt", {k: round(gbt_metrics[label][k], 6) for k in ("route_wape_all", "route_wape_y1", "route_wape_y2", "network_wape", "bias_all", "network_mae", "network_rmse")})
